<a href="https://colab.research.google.com/github/NishantAhirwar07/Crop_Demand_Forecasting/blob/main/New_crop_forecasting(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crop Demand & Price Forecasting and route optimization.

##Cell 01 : Install Important libraries

In [ ]:
# ============================================================
# CELL 1: INSTALL REQUIRED LIBRARIES
# ============================================================

# XGBoost = main machine learning algorithm
# scikit-learn = preprocessing, train/test split, evaluation
# pandas = data handling
# numpy = numerical calculations
# matplotlib/seaborn = visualization
# joblib = save trained ML models

!pip install -q xgboost pandas numpy scikit-learn matplotlib seaborn joblib

---

### Cell 2 — Import libraries

In [ ]:
# ============================================================
# CELL 2: IMPORT IMPORTANT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

# Machine Learning
from xgboost import XGBRegressor

# Data preprocessing
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Evaluation metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Save/load ML models
import joblib

# Display settings
pd.set_option("display.max_columns", None)

print("All libraries imported successfully!")

All libraries imported successfully!


---

# Cell 3 — Upload your dataset

In [ ]:
# ============================================================
# CELL 3: UPLOAD CSV DATASET
# ============================================================

from google.colab import files

# Open the file upload window
uploaded = files.upload()

print("File uploaded successfully!")

After running this cell, select:

```text
soybean_madhya_pradesh_10000_enhanced.csv
```

---

# Cell 4 — Load dataset

In [ ]:
# ============================================================
# CELL 4: LOAD DATASET
# ============================================================

# Get the uploaded filename automatically
file_name = list(uploaded.keys())[0]

# Read CSV
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")

# Display first 5 rows
display(df.head())

---

# Cell 5 — Understand the dataset

In [ ]:
# ============================================================
# CELL 5: BASIC DATASET INFORMATION
# ============================================================

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

---

# Cell 6 — Check missing values

In [ ]:
# ============================================================
# CELL 6: CHECK MISSING VALUES
# ============================================================

missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

print("\nTotal missing values:", df.isnull().sum().sum())

If your output is:

```text
Total missing values: 0
```

then your dataset doesn't contain missing values.

---

# Cell 7 — Statistical analysis

In [ ]:
# ============================================================
# CELL 7: STATISTICAL SUMMARY
# ============================================================

# describe() gives:
# count
# mean
# standard deviation
# minimum
# maximum
# etc.

display(df.describe())

---

# Cell 8 — Check categorical columns

In [ ]:
# ============================================================
# CELL 8: IDENTIFY CATEGORICAL AND NUMERICAL FEATURES
# ============================================================

# Categorical columns
categorical_columns = df.select_dtypes(
    include=["object"]
).columns.tolist()

# Numerical columns
numerical_columns = df.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

print("\nNumerical columns:")
print(numerical_columns)

You should see categorical columns similar to:

```text
crop
district
market
Season
Weather
Festival
```

depending on your dataset.

---

# Cell 9 — Remove unnecessary columns

Before training, we don't want ID-type columns or columns that shouldn't be used as predictors.

In [ ]:
# ============================================================
# CELL 9: REMOVE UNNECESSARY COLUMNS
# ============================================================

# Make a copy so original dataset remains unchanged
data = df.copy()

# Columns that are generally identifiers rather than useful
# predictive variables.
possible_id_columns = [
    "id",
    "ID",
    "date_id"
]

# Remove only columns that actually exist
columns_to_remove = [
    col for col in possible_id_columns
    if col in data.columns
]

data.drop(columns=columns_to_remove, inplace=True)

print("Removed columns:", columns_to_remove)

print("\nRemaining columns:")
print(data.columns.tolist())

---

# Cell 10 — Define targets

This is a very important step.

We have **two different prediction problems**.

### Target 1

```text
Demand
```

### Target 2

```text
Price
```

In [ ]:
# ============================================================
# CELL 10: DEFINE TARGET VARIABLES
# ============================================================

# Demand prediction target
DEMAND_TARGET = "demand_kg"

# Price prediction target
PRICE_TARGET = "mandi_price_per_kg"

# Check that both targets exist
if DEMAND_TARGET not in data.columns:
    raise ValueError(f"{DEMAND_TARGET} not found in dataset!")

if PRICE_TARGET not in data.columns:
    raise ValueError(f"{PRICE_TARGET} not found in dataset!")

print("Demand target:", DEMAND_TARGET)
print("Price target:", PRICE_TARGET)

---

# 11. Demand Prediction Model

Now we build our first XGBoost model.

## Features

We should NOT use the target itself as an input.

In [ ]:
# ============================================================
# CELL 11: CREATE DEMAND DATASET
# ============================================================

# Copy dataset
demand_data = data.copy()

# Demand target
y_demand = demand_data[DEMAND_TARGET]

# Remove target columns from input features
# Price is also removed here to avoid creating a direct
# target leakage relationship with the current price.
X_demand = demand_data.drop(
    columns=[
        DEMAND_TARGET,
        PRICE_TARGET
    ],
    errors="ignore"
)

print("Demand feature shape:", X_demand.shape)
print("Demand target shape:", y_demand.shape)

display(X_demand.head())

---

# 12. Detect categorical features

In [ ]:
# ============================================================
# CELL 12: DEMAND MODEL - IDENTIFY FEATURE TYPES
# ============================================================

demand_categorical = X_demand.select_dtypes(
    include=["object"]
).columns.tolist()

demand_numerical = X_demand.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical features:")
print(demand_categorical)

print("\nNumerical features:")
print(demand_numerical)

---

# 13. Train/test split

In [ ]:
# ============================================================
# CELL 13: TRAIN-TEST SPLIT FOR DEMAND MODEL
# ============================================================

X_train_demand, X_test_demand, y_train_demand, y_test_demand = train_test_split(
    X_demand,
    y_demand,
    test_size=0.20,       # 80% training, 20% testing
    random_state=42
)

print("Training samples:", len(X_train_demand))
print("Testing samples:", len(X_test_demand))

---

# 14. One-Hot Encoding

XGBoost cannot directly understand text such as:

```text
Kharif Harvest
Rainy
Diwali
```

So we convert categorical values into numerical values.

In [ ]:
# ============================================================
# CELL 14: PREPROCESS DEMAND FEATURES
# ============================================================

demand_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            demand_categorical
        ),
        (
            "numerical",
            "passthrough",
            demand_numerical
        )
    ]
)

# Fit preprocessing on training data
X_train_demand_processed = demand_preprocessor.fit_transform(
    X_train_demand
)

# Transform testing data
X_test_demand_processed = demand_preprocessor.transform(
    X_test_demand
)

print(
    "Processed training shape:",
    X_train_demand_processed.shape
)

print(
    "Processed testing shape:",
    X_test_demand_processed.shape
)

---

# 15. Train XGBoost Demand Model ⭐

In [ ]:
# ============================================================
# CELL 15: TRAIN XGBOOST DEMAND MODEL
# ============================================================

demand_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,

    # Objective for numerical regression
    objective="reg:squarederror",

    # Reproducibility
    random_state=42,

    # CPU training
    n_jobs=-1
)

# Train model
demand_model.fit(
    X_train_demand_processed,
    y_train_demand
)

print("XGBoost Demand Model trained successfully!")

---

# 16. Demand prediction

In [ ]:
# ============================================================
# CELL 16: PREDICT DEMAND
# ============================================================

# Predict demand for test data
demand_predictions = demand_model.predict(
    X_test_demand_processed
)

# Display first predictions
comparison_demand = pd.DataFrame({
    "Actual Demand": y_test_demand.values,
    "Predicted Demand": demand_predictions
})

display(comparison_demand.head(10))

---

# 17. Evaluate Demand Model

In [ ]:
# ============================================================
# CELL 17: DEMAND MODEL EVALUATION
# ============================================================

mae_demand = mean_absolute_error(
    y_test_demand,
    demand_predictions
)

rmse_demand = np.sqrt(
    mean_squared_error(
        y_test_demand,
        demand_predictions
    )
)

r2_demand = r2_score(
    y_test_demand,
    demand_predictions
)

print("================================")
print("DEMAND MODEL PERFORMANCE")
print("================================")

print(f"MAE  : {mae_demand:.2f} kg")
print(f"RMSE : {rmse_demand:.2f} kg")
print(f"R²   : {r2_demand:.4f}")

### Understanding these metrics

```text
MAE
↓
Average prediction error

RMSE
↓
Penalizes large errors

R²
↓
How well model explains variation
```

For example:

```text
R² = 0.92
```

would mean the model explains approximately 92% of the variation in the target on that test set.

---

# 18. Demand prediction graph

In [ ]:
# ============================================================
# CELL 18: ACTUAL VS PREDICTED DEMAND
# ============================================================

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test_demand,
    demand_predictions,
    alpha=0.5
)

# Perfect prediction line
minimum = min(
    y_test_demand.min(),
    demand_predictions.min()
)

maximum = max(
    y_test_demand.max(),
    demand_predictions.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel("Actual Demand (kg)")
plt.ylabel("Predicted Demand (kg)")
plt.title("XGBoost: Actual vs Predicted Demand")

plt.show()

The closer the points are to the diagonal line, the better the predictions.

---

# 19. Price Prediction Model

Now we create the second XGBoost model.

There is one important design decision here.

We want the price model to know the **predicted demand**.

So we will create predicted demand for all observations using the demand model.

For a more rigorous production pipeline, you would generate these values using out-of-fold predictions during training to avoid leakage. For a first Colab prototype, we'll keep the implementation straightforward.

In [ ]:
# ============================================================
# CELL 19: CREATE DATASET FOR PRICE MODEL
# ============================================================

price_data = data.copy()

# Create predicted demand for every row
all_demand_processed = demand_preprocessor.transform(
    X_demand
)

predicted_demand_all = demand_model.predict(
    all_demand_processed
)

# Add predicted demand as a feature
price_data["Predicted Demand"] = predicted_demand_all

# Target
y_price = price_data[PRICE_TARGET]

# Features
X_price = price_data.drop(
    columns=[
        PRICE_TARGET,
        DEMAND_TARGET
    ],
    errors="ignore"
)

print("Price feature shape:", X_price.shape)
print("Price target shape:", y_price.shape)

---

# 20. Price model feature types

In [ ]:
# ============================================================
# CELL 20: PRICE MODEL FEATURE TYPES
# ============================================================

price_categorical = X_price.select_dtypes(
    include=["object"]
).columns.tolist()

price_numerical = X_price.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical features:")
print(price_categorical)

print("\nNumerical features:")
print(price_numerical)

---

# 21. Train/test split for price

In [ ]:
# ============================================================
# CELL 21: PRICE TRAIN-TEST SPLIT
# ============================================================

X_train_price, X_test_price, y_train_price, y_test_price = train_test_split(
    X_price,
    y_price,
    test_size=0.20,
    random_state=42
)

print("Price training samples:", len(X_train_price))
print("Price testing samples:", len(X_test_price))

---

# 22. Preprocess price features

In [ ]:
# ============================================================
# CELL 22: PREPROCESS PRICE FEATURES
# ============================================================

price_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            price_categorical
        ),
        (
            "numerical",
            "passthrough",
            price_numerical
        )
    ]
)

X_train_price_processed = price_preprocessor.fit_transform(
    X_train_price
)

X_test_price_processed = price_preprocessor.transform(
    X_test_price
)

print(
    "Processed price training shape:",
    X_train_price_processed.shape
)

print(
    "Processed price testing shape:",
    X_test_price_processed.shape
)

---

# 23. Train XGBoost Price Model ⭐

In [ ]:
# ============================================================
# CELL 23: TRAIN XGBOOST PRICE MODEL
# ============================================================

price_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,

    objective="reg:squarederror",

    random_state=42,

    n_jobs=-1
)

price_model.fit(
    X_train_price_processed,
    y_train_price
)

print("XGBoost Price Model trained successfully!")

---

# 24. Predict price

In [ ]:
# ============================================================
# CELL 24: PREDICT PRICE
# ============================================================

price_predictions = price_model.predict(
    X_test_price_processed
)

price_comparison = pd.DataFrame({
    "Actual Price": y_test_price.values,
    "Predicted Price": price_predictions
})

display(price_comparison.head(10))

---

# 25. Evaluate Price Model

In [ ]:
# ============================================================
# CELL 25: PRICE MODEL EVALUATION
# ============================================================

mae_price = mean_absolute_error(
    y_test_price,
    price_predictions
)

rmse_price = np.sqrt(
    mean_squared_error(
        y_test_price,
        price_predictions
    )
)

r2_price = r2_score(
    y_test_price,
    price_predictions
)

print("================================")
print("PRICE MODEL PERFORMANCE")
print("================================")

print(f"MAE  : ₹{mae_price:.2f}/kg")
print(f"RMSE : ₹{rmse_price:.2f}/kg")
print(f"R²   : {r2_price:.4f}")

---

# 26. Price prediction graph

In [ ]:
# ============================================================
# CELL 26: ACTUAL VS PREDICTED PRICE
# ============================================================

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test_price,
    price_predictions,
    alpha=0.5
)

minimum = min(
    y_test_price.min(),
    price_predictions.min()
)

maximum = max(
    y_test_price.max(),
    price_predictions.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel("Actual Price (₹/kg)")
plt.ylabel("Predicted Price (₹/kg)")
plt.title("XGBoost: Actual vs Predicted Price")

plt.show()

---

# 27. Feature importance

This is useful for your hackathon presentation because you can show **what factors influence price and demand**.

In [ ]:
# ============================================================
# CELL 27: DEMAND FEATURE IMPORTANCE
# ============================================================

# Get names of encoded features
feature_names_demand = demand_preprocessor.get_feature_names_out()

importance_demand = pd.DataFrame({
    "Feature": feature_names_demand,
    "Importance": demand_model.feature_importances_
})

importance_demand = importance_demand.sort_values(
    by="Importance",
    ascending=False
)

display(importance_demand.head(15))

---

# 28. Plot demand feature importance

In [ ]:
# ============================================================
# CELL 28: PLOT DEMAND FEATURE IMPORTANCE
# ============================================================

top_features = importance_demand.head(15)

plt.figure(figsize=(10, 6))

plt.barh(
    top_features["Feature"][::-1],
    top_features["Importance"][::-1]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top Features Influencing Crop Demand")

plt.show()

---

# 29. Price feature importance

In [ ]:
# ============================================================
# CELL 29: PRICE FEATURE IMPORTANCE
# ============================================================

feature_names_price = price_preprocessor.get_feature_names_out()

importance_price = pd.DataFrame({
    "Feature": feature_names_price,
    "Importance": price_model.feature_importances_
})

importance_price = importance_price.sort_values(
    by="Importance",
    ascending=False
)

display(importance_price.head(15))

---

# 30. Save your models

You will eventually use these models in a Streamlit application.

In [ ]:
# ============================================================
# CELL 30: SAVE TRAINED MODELS
# ============================================================

# Save demand model
joblib.dump(
    demand_model,
    "xgboost_demand_model.pkl"
)

# Save price model
joblib.dump(
    price_model,
    "xgboost_price_model.pkl"
)

# Save preprocessing objects
joblib.dump(
    demand_preprocessor,
    "demand_preprocessor.pkl"
)

joblib.dump(
    price_preprocessor,
    "price_preprocessor.pkl"
)

print("All models and preprocessors saved!")

---

# 31. Download models

In [ ]:
# ============================================================
# CELL 31: DOWNLOAD MODELS
# ============================================================

from google.colab import files

files.download("xgboost_demand_model.pkl")
files.download("xgboost_price_model.pkl")

files.download("demand_preprocessor.pkl")
files.download("price_preprocessor.pkl")

---

# 32. Create prediction function

Now we create the part that your future Streamlit app can use.

In [ ]:
# ============================================================
# CELL 32: COMPLETE DEMAND + PRICE PREDICTION FUNCTION
# ============================================================

def predict_crop_demand_price(input_data):
    """
    Predict crop demand and market price.

    input_data:
        pandas DataFrame containing one or more observations.

    Returns:
        Predicted demand and predicted price.
    """

    # -----------------------------
    # STEP 1: Demand prediction
    # -----------------------------

    demand_input = input_data.drop(
        columns=[
            DEMAND_TARGET,
            PRICE_TARGET,
            "Predicted Demand"
        ],
        errors="ignore"
    )

    demand_processed = demand_preprocessor.transform(
        demand_input
    )

    predicted_demand = demand_model.predict(
        demand_processed
    )

    # -----------------------------
    # STEP 2: Add predicted demand
    # -----------------------------

    price_input = input_data.drop(
        columns=[
            DEMAND_TARGET,
            PRICE_TARGET
        ],
        errors="ignore"
    ).copy()

    price_input["Predicted Demand"] = predicted_demand

    # -----------------------------
    # STEP 3: Price prediction
    # -----------------------------

    price_processed = price_preprocessor.transform(
        price_input
    )

    predicted_price = price_model.predict(
        price_processed
    )

    # -----------------------------
    # STEP 4: Return results
    # -----------------------------

    result = pd.DataFrame({
        "Predicted Demand (kg)": predicted_demand,
        "Predicted Price (₹/kg)": predicted_price
    })

    return result

---

# 33. Test prediction

In [ ]:
# ============================================================
# CELL 33: TEST THE PREDICTION SYSTEM
# ============================================================

# Take one existing row as an example
sample_input = data.drop(
    columns=[
        DEMAND_TARGET,
        PRICE_TARGET
    ],
    errors="ignore"
).iloc[[0]].copy()

# Predict
prediction = predict_crop_demand_price(
    sample_input
)

display(prediction)

You should get something like:

```text
Predicted Demand     Predicted Price
------------------------------------
2850 kg              ₹63.40/kg
```

Your actual numbers will depend on your trained model.

---

# 34. Best Market Recommendation Engine

Now comes the **most important part of your project**.

The ML models predict:

```text
Demand
Price
```

Then an optimization layer determines:

```text
BEST MARKET
```

For this, your dataset needs market-specific information.

If your dataset contains:

```text
market
```

we can evaluate each market.

The basic calculation is:

```text
Gross Revenue
      =
Selling Quantity × Predicted Price

Net Revenue
      =
Gross Revenue
− Transportation Cost
− Market Fees
```

---

## Cell 34 — Market scoring function

In [ ]:
# ============================================================
# CELL 34: MARKET RECOMMENDATION FUNCTION
# ============================================================

def calculate_market_profit(
    selling_quantity,
    predicted_price,
    transport_cost,
    market_fee=0
):
    """
    Calculate expected revenue and net profit
    for selling crop at a particular market.
    """

    # Total money received before expenses
    gross_revenue = (
        selling_quantity * predicted_price
    )

    # Final estimated profit after expenses
    net_profit = (
        gross_revenue
        - transport_cost
        - market_fee
    )

    return gross_revenue, net_profit

---

# 35. Example market comparison

In [ ]:
# ============================================================
# CELL 35: EXAMPLE MARKET COMPARISON
# ============================================================

selling_quantity = 2000  # kg

markets = pd.DataFrame({
    "Market": [
        "Gwalior",
        "Morena",
        "Bhopal",
        "Indore"
    ],

    "Predicted Price": [
        63,
        65,
        61,
        59
    ],

    "Predicted Demand": [
        3500,
        2900,
        4000,
        4500
    ],

    "Transport Cost": [
        3000,
        5000,
        15000,
        18000
    ],

    "Market Fee": [
        1000,
        1000,
        1500,
        2000
    ]
})

# Calculate gross revenue
markets["Gross Revenue"] = (
    selling_quantity *
    markets["Predicted Price"]
)

# Calculate net profit
markets["Net Profit"] = (
    markets["Gross Revenue"]
    - markets["Transport Cost"]
    - markets["Market Fee"]
)

# Sort markets by highest profit
markets = markets.sort_values(
    by="Net Profit",
    ascending=False
)

display(markets)

---

# 36. Automatically select best market

In [ ]:
# ============================================================
# CELL 36: SELECT BEST MARKET
# ============================================================

best_market = markets.iloc[0]

print("======================================")
print("🏆 BEST MARKET RECOMMENDATION")
print("======================================")

print("Market:",
      best_market["Market"])

print(
    "Predicted Price: ₹",
    round(best_market["Predicted Price"], 2),
    "/kg"
)

print(
    "Predicted Demand:",
    round(best_market["Predicted Demand"], 2),
    "kg"
)

print(
    "Expected Gross Revenue: ₹",
    round(best_market["Gross Revenue"], 2)
)

print(
    "Expected Net Profit: ₹",
    round(best_market["Net Profit"], 2)
)

---

# 37. Your final system

After completing the notebook, your architecture becomes:

```text
                  CSV DATASET
                       │
                       ▼
              DATA PREPROCESSING
                       │
                       ▼
              FEATURE ENGINEERING
                       │
          ┌────────────┴────────────┐
          │                         │
          ▼                         ▼
   XGBOOST DEMAND             XGBOOST PRICE
       MODEL                     MODEL
          │                         │
          ▼                         ▼
 Predicted Demand            Predicted Price
          │                         │
          └────────────┬────────────┘
                       │
                       ▼
              MARKET OPTIMIZER
                       │
                       ▼
            Revenue Calculation
                       │
                       ▼
             Transportation Cost
                       │
                       ▼
                Market Fee
                       │
                       ▼
                 NET PROFIT
                       │
                       ▼
              🏆 BEST MARKET
```

## One important improvement before calling this a final model

Your current dataset is excellent for **building the prototype**, but because some of the added columns were synthetically generated, the model can learn artificial relationships.

For a serious hackathon/real-world version, the next step should be:

```text
Real historical data
       ↓
Date + Market + District
       ↓
Historical Demand
Historical Price
Historical Stock
Weather
Festival
Transportation
       ↓
Lag Features
       ↓
XGBoost
       ↓
Demand + Price
       ↓
Market Optimization
```

Also, **do not evaluate the final model only on the same synthetic data used to generate the features**. A real held-out time period or real market data would give you a much more credible accuracy estimate.